# Expanded Q4 2025 AI/Tech Earnings Eval

This notebook runs or loads cached outputs for the expanded AI/Tech Q4 2025 earnings universe. Agent outputs are saved under `data/agent_outputs/q4_2025_ai_tech/` by agent type, so future runs can reuse them without calling the models again.


In [41]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "pyproject.toml").exists() else CURRENT_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
load_dotenv(REPO_ROOT / ".env", override=True)

from openclam.evaluation import q4_earnings_cache as q4
from openclam.agents.news_macro import news_macro_agent

print(f"Repo root: {REPO_ROOT}")
print(f"Cache root: {REPO_ROOT / q4.DEFAULT_CACHE_ROOT}")
print(f"VERTEX_PROJECT loaded: {bool(os.getenv('VERTEX_PROJECT') or os.getenv('GOOGLE_CLOUD_PROJECT'))}")
print(f"OPENAI_API_KEY loaded: {bool(os.getenv('OPENAI_API_KEY'))}")


Repo root: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory
Cache root: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech
VERTEX_PROJECT loaded: True
OPENAI_API_KEY loaded: False


## Configure Universe and Cache

Set `RUN_AGENTS=True` when you want to generate missing cached outputs. Keep `FORCE_RERUN=False` to reuse files that already exist.


In [42]:
import importlib
from openclam.evaluation import q4_earnings_cache as q4

q4 = importlib.reload(q4)

print(q4.__file__)
print(hasattr(q4, "q4_2025_combined_cio_advantage_df"))


/Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/src/openclam/evaluation/q4_earnings_cache.py
True


In [43]:
# Full combined Q4 2025 AI/Tech + CIO-advantage universe.
# This uses all tickers defined in q4_earnings_cache.py.
earnings_df = q4.q4_2025_combined_cio_advantage_df()

CACHE_ROOT = REPO_ROOT / q4.DEFAULT_CACHE_ROOT
RUN_AGENTS = True
FORCE_RERUN = False  # keep False: cached names will not rerun; only missing tickers are generated

PRE_TRADING_DAYS = 7
POST_TRADING_DAYS = 7
LONG_POST_TRADING_DAYS = 30
NEUTRAL_BAND = 0.02

VERTEX_PROJECT = os.getenv("VERTEX_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-2.5-flash")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
NEWS_MODEL = os.getenv("NEWS_MODEL", "gpt-5.4-nano")
LLM_PROVIDER = "vertex" if VERTEX_PROJECT else "openai"

print("Full universe size:", len(earnings_df))
earnings_df


Full universe size: 55


,ticker,company,earnings_date,bucket
0,AAPL,Apple,2026-01-29,mega_cap_platform
1,MSFT,Microsoft,2026-01-28,mega_cap_platform
2,GOOGL,Alphabet,2026-02-03,mega_cap_platform
3,AMZN,Amazon,2026-02-05,mega_cap_platform
4,META,Meta Platforms,2026-01-28,mega_cap_platform
5,TSLA,Tesla,2026-01-28,mega_cap_platform
6,NVDA,Nvidia,2026-02-25,mega_cap_platform
7,AMD,Advanced Micro Devices,2026-02-03,ai_semis
8,AVGO,Broadcom,2026-03-05,ai_semis
9,MU,Micron Technology,2025-12-17,ai_semis


## Build Price Eval Table


In [44]:
summary_df, paths_df = q4.build_price_tables(
    earnings_df=earnings_df,
    cache_root=CACHE_ROOT,
    pre_trading_days=PRE_TRADING_DAYS,
    post_trading_days=POST_TRADING_DAYS,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
)

news_macro_agent.format_return_columns(summary_df)


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,spy_post_30d_return,abnormal_30d_vs_spy,qqq_post_7d_return,abnormal_vs_qqq,qqq_post_30d_return,abnormal_30d_vs_qqq,realized_direction_vs_qqq,realized_30d_direction_vs_qqq,long_horizon_trading_days,bucket
0,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,-4.57%,1.51%,-2.40%,8.83%,-5.67%,2.60%,up,up,30,mega_cap_platform
1,MSFT,Microsoft,2026-01-28,4.73%,-9.99%,-12.10%,-16.71%,-16.37%,-12.77%,-12.41%,...,-4.22%,-12.15%,-3.72%,-12.99%,-5.68%,-10.69%,down,down,30,mega_cap_platform
2,GOOGL,Alphabet,2026-02-03,3.59%,-1.96%,-4.96%,-9.04%,-9.36%,-5.77%,-6.11%,...,-4.08%,-5.29%,-2.58%,-6.46%,-3.51%,-5.86%,down,down,30,mega_cap_platform
3,AMZN,Amazon,2026-02-05,-8.99%,-5.55%,-7.06%,-9.67%,-7.78%,-17.79%,-16.07%,...,-4.03%,-3.75%,0.72%,-10.39%,-2.51%,-5.27%,down,down,30,mega_cap_platform
4,META,Meta Platforms,2026-01-28,7.82%,10.40%,5.63%,-1.09%,-4.57%,6.64%,2.89%,...,-4.22%,-0.35%,-3.72%,2.64%,-5.68%,1.11%,up,up,30,mega_cap_platform
5,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,-4.22%,-4.23%,-3.72%,-0.99%,-5.68%,-2.77%,down,down,30,mega_cap_platform
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,-1.64%,-4.31%,-2.75%,-6.33%,-0.93%,-5.02%,down,down,30,mega_cap_platform
7,AMD,Advanced Micro Devices,2026-02-03,-6.77%,-17.31%,-13.91%,-14.94%,-17.62%,-20.69%,-23.19%,...,-4.08%,-13.54%,-2.58%,-12.36%,-3.51%,-14.11%,down,down,30,ai_semis
8,AVGO,Broadcom,2026-03-05,2.24%,-0.69%,2.95%,-2.36%,22.42%,-0.18%,25.16%,...,4.52%,17.91%,-1.40%,-0.96%,6.69%,15.73%,down,up,30,ai_semis
9,MU,Micron Technology,2025-12-17,-8.67%,10.21%,22.65%,30.58%,94.21%,19.26%,77.38%,...,3.88%,90.33%,3.54%,27.04%,4.42%,89.79%,up,up,30,ai_semis


## Run or Load Cached Agent Outputs


In [47]:
if RUN_AGENTS:
    packets_by_ticker, errors_by_ticker = q4.run_q4_2025_universe_agents(
        earnings_df=earnings_df,
        cache_root=CACHE_ROOT,
        force=FORCE_RERUN,
        lookback_days=14,
        max_news=10,
        llm_provider=LLM_PROVIDER,
        vertex_project=VERTEX_PROJECT,
        vertex_location=VERTEX_LOCATION,
        vertex_model=VERTEX_MODEL,
        openai_model=OPENAI_MODEL,
        news_model=NEWS_MODEL,
    )
else:
    packets_by_ticker = q4.load_cached_packets_by_ticker(CACHE_ROOT)
    errors_by_ticker = {}

print("cached/available tickers:", sorted(packets_by_ticker))
errors_by_ticker


cached/available tickers: ['AAPL', 'ADBE', 'ALAB', 'AMAT', 'AMD', 'AMT', 'AMZN', 'ANET', 'APP', 'ARM', 'ASML', 'AVGO', 'CEG', 'CLS', 'COHR', 'CORZ', 'CRM', 'CRWD', 'DDOG', 'DELL', 'DLR', 'EQIX', 'ETN', 'FLEX', 'GEV', 'GOOGL', 'IREN', 'IRM', 'KLAC', 'LRCX', 'MDB', 'META', 'MRVL', 'MSFT', 'MU', 'NET', 'NOW', 'NRG', 'NVDA', 'OKTA', 'ORCL', 'PANW', 'PLTR', 'PWR', 'QCOM', 'SHOP', 'SMCI', 'SNOW', 'STX', 'TEAM', 'TSLA', 'TSM', 'VRT', 'WDC', 'ZS']


{'AAPL': {},
 'MSFT': {},
 'GOOGL': {},
 'AMZN': {},
 'META': {},
 'TSLA': {},
 'NVDA': {},
 'AMD': {},
 'AVGO': {},
 'MU': {},
 'QCOM': {},
 'ARM': {},
 'TSM': {},
 'ASML': {},
 'AMAT': {},
 'LRCX': {},
 'KLAC': {},
 'ORCL': {},
 'DELL': {},
 'SMCI': {},
 'ANET': {},
 'VRT': {},
 'CRM': {},
 'ADBE': {},
 'NOW': {},
 'SNOW': {},
 'PLTR': {},
 'DDOG': {},
 'MDB': {},
 'NET': {},
 'MRVL': {},
 'WDC': {},
 'STX': {},
 'COHR': {},
 'ALAB': {},
 'ETN': {},
 'PWR': {},
 'CEG': {},
 'NRG': {},
 'GEV': {},
 'EQIX': {},
 'DLR': {},
 'IRM': {},
 'AMT': {},
 'CORZ': {},
 'IREN': {},
 'CLS': {},
 'FLEX': {},
 'TEAM': {},
 'ZS': {},
 'CRWD': {},
 'PANW': {},
 'OKTA': {},
 'APP': {},
 'SHOP': {}}

## Run CIO Eval From Cached Packets


In [48]:
expected_tickers = set(summary_df["ticker"].str.upper())

cached_cio_files = {
    path.stem.upper()
    for path in (CACHE_ROOT / "cio").glob("*.json")
}

missing_cio_tickers = sorted(expected_tickers - cached_cio_files)

print("Missing CIO tickers:", missing_cio_tickers)

incremental_results = {}

for ticker in missing_cio_tickers:
    print("\n" + "=" * 80)
    print(f"Running CIO for {ticker}...")
    print("=" * 80)

    one_summary_df = summary_df[
        summary_df["ticker"].str.upper() == ticker
    ].copy()

    one_packets = {
        ticker: packets_by_ticker.get(ticker, [])
    }

    if not one_packets[ticker]:
        print(f"Skipped {ticker}: no packets found.")
        continue

    try:
        one_cio_eval, one_cio_results, one_cio_summary = q4.run_cached_cio_eval(
            one_summary_df,
            one_packets,
            cache_root=CACHE_ROOT,
            long_post_trading_days=LONG_POST_TRADING_DAYS,
            neutral_band=NEUTRAL_BAND,
            use_llm_debate=True,
            use_llm_decision=True,
            llm_provider=LLM_PROVIDER,
            debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
            decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
            vertex_project=VERTEX_PROJECT,
            vertex_location=VERTEX_LOCATION,
        )

        incremental_results[ticker] = one_cio_results.get(ticker)
        print(f"Saved CIO result for {ticker}: {CACHE_ROOT / 'cio' / f'{ticker}.json'}")

    except Exception as exc:
        print(f"Failed CIO for {ticker}: {repr(exc)}")


Missing CIO tickers: ['ADBE', 'ARM', 'DDOG', 'KLAC', 'MDB', 'NET', 'NOW', 'QCOM', 'SMCI', 'SNOW']

Running CIO for ADBE...
Saved CIO result for ADBE: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/ADBE.json

Running CIO for ARM...
Saved CIO result for ARM: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/ARM.json

Running CIO for DDOG...
Saved CIO result for DDOG: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/DDOG.json

Running CIO for KLAC...
Saved CIO result for KLAC: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/KLAC.json

Running CIO for MDB...
Saved CIO result for MDB: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/MDB.json

Running CIO for NET...
Saved CIO result for NET: /Users/zhang/Github/OpenClam_Multi-Agent_Investm

In [49]:
# Rebuild full cio_eval from all cached CIO workflows for the current summary_df universe

expected_tickers = set(summary_df["ticker"].str.upper())

cio_results = {
    path.stem.upper(): q4.load_json(path)
    for path in (CACHE_ROOT / "cio").glob("*.json")
    if path.stem.upper() in expected_tickers
}

print("Expected tickers:", len(expected_tickers))
print("Loaded cached CIO workflows:", len(cio_results))
print("Missing cached CIO workflows:", sorted(expected_tickers - set(cio_results)))

def stance_to_direction(stance):
    if stance == "Bullish":
        return "up"
    if stance == "Bearish":
        return "down"
    return None

def match_direction(predicted, realized, abnormal, neutral_band=0.02):
    if pd.isna(abnormal) or realized is None:
        return None
    abnormal = float(abnormal)
    if predicted is None:
        return abs(abnormal) <= neutral_band
    return predicted == realized

def match_reason(predicted, realized, abnormal, neutral_band=0.02):
    if pd.isna(abnormal) or realized is None:
        return "missing realized abnormal return"
    abnormal = float(abnormal)
    if predicted is None:
        if abs(abnormal) <= neutral_band:
            return "neutral matched: abnormal return stayed inside neutral band"
        return "neutral missed: abnormal return moved outside neutral band"
    return "matched" if predicted == realized else "missed"

long_abnormal_col = f"abnormal_{LONG_POST_TRADING_DAYS}d_vs_qqq"
long_direction_col = f"realized_{LONG_POST_TRADING_DAYS}d_direction_vs_qqq"

rows = []

for _, row in summary_df.iterrows():
    ticker = row["ticker"].upper()
    workflow = cio_results.get(ticker)
    base = row.to_dict()

    if not workflow:
        base.update({
            "cio_ready": False,
            "cio_short_term_stance": None,
            "cio_long_term_stance": None,
            "cio_action": None,
            "cio_confidence": None,
            "cio_debate_triggered": None,
            "cio_debate_conflict_level": None,
            "cio_reason": "missing cached CIO workflow",
            "cio_short_direction_match": None,
            "cio_short_direction_match_reason": "missing cached CIO workflow",
            "cio_long_direction_match": None,
            "cio_long_direction_match_reason": "missing cached CIO workflow",
        })
        rows.append(base)
        continue

    decision = workflow["final_decision"]
    debate = workflow["debate"]

    short_stance = decision["final_short_term_stance"]
    long_stance = decision["final_long_term_stance"]

    short_pred = stance_to_direction(short_stance)
    long_pred = stance_to_direction(long_stance)

    base.update({
        "cio_ready": True,
        "cio_short_term_stance": short_stance,
        "cio_long_term_stance": long_stance,
        "cio_action": decision.get("investment_action"),
        "cio_confidence": decision.get("confidence"),
        "cio_debate_triggered": debate.get("triggered"),
        "cio_debate_conflict_level": debate.get("trigger", {}).get("conflict_level"),
        "cio_reason": decision.get("reason_for_final_decision"),
        "cio_short_direction_match": match_direction(
            short_pred,
            row.get("realized_direction_vs_qqq"),
            row.get("abnormal_vs_qqq"),
            NEUTRAL_BAND,
        ),
        "cio_short_direction_match_reason": match_reason(
            short_pred,
            row.get("realized_direction_vs_qqq"),
            row.get("abnormal_vs_qqq"),
            NEUTRAL_BAND,
        ),
        "cio_long_direction_match": match_direction(
            long_pred,
            row.get(long_direction_col),
            row.get(long_abnormal_col),
            NEUTRAL_BAND,
        ),
        "cio_long_direction_match_reason": match_reason(
            long_pred,
            row.get(long_direction_col),
            row.get(long_abnormal_col),
            NEUTRAL_BAND,
        ),
    })

    rows.append(base)

cio_eval = pd.DataFrame(rows)

cio_summary = {
    "overall": {
        "cases": int(len(cio_eval)),
        "cio_ready_cases": int((cio_eval["cio_ready"] == True).sum()),
        "debate_trigger_rate": float((cio_eval["cio_debate_triggered"] == True).mean()),
        "cio_short_direction_match_evaluable": int(cio_eval["cio_short_direction_match"].notna().sum()),
        "cio_short_direction_match_matched": int((cio_eval["cio_short_direction_match"] == True).sum()),
        "cio_short_direction_match_accuracy": float(
            (cio_eval[cio_eval["cio_short_direction_match"].notna()]["cio_short_direction_match"] == True).mean()
        ),
        "cio_long_direction_match_evaluable": int(cio_eval["cio_long_direction_match"].notna().sum()),
        "cio_long_direction_match_matched": int((cio_eval["cio_long_direction_match"] == True).sum()),
        "cio_long_direction_match_accuracy": float(
            (cio_eval[cio_eval["cio_long_direction_match"].notna()]["cio_long_direction_match"] == True).mean()
        ),
    },
    "by_bucket": q4.summarize_cio_eval_by_bucket(cio_eval),
}

cio_eval.to_csv(CACHE_ROOT / "tables" / "cio_eval_combined_cache_aware.csv", index=False)
q4.save_json(CACHE_ROOT / "tables" / "cio_eval_combined_cache_aware_summary.json", cio_summary)

print(cio_summary)
news_macro_agent.format_return_columns(cio_eval)


Expected tickers: 55
Loaded cached CIO workflows: 55
Missing cached CIO workflows: []
{'overall': {'cases': 55, 'cio_ready_cases': 55, 'debate_trigger_rate': 0.6, 'cio_short_direction_match_evaluable': 55, 'cio_short_direction_match_matched': 20, 'cio_short_direction_match_accuracy': 0.36363636363636365, 'cio_long_direction_match_evaluable': 55, 'cio_long_direction_match_matched': 30, 'cio_long_direction_match_accuracy': 0.5454545454545454}, 'by_bucket': {'ai_infrastructure': {'cases': 9, 'debate_trigger_rate': 0.6666666666666666, 'short_accuracy': 0.3333333333333333, 'long_accuracy': 0.5555555555555556, 'avg_confidence': 0.7594444444444445}, 'ai_semis': {'cases': 13, 'debate_trigger_rate': 0.3076923076923077, 'short_accuracy': 0.38461538461538464, 'long_accuracy': 0.6153846153846154, 'avg_confidence': 0.8230769230769232}, 'data_center_operator': {'cases': 2, 'debate_trigger_rate': 0.5, 'short_accuracy': 0.0, 'long_accuracy': 0.5, 'avg_confidence': 0.7}, 'data_center_reit': {'cases': 4

,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,cio_long_term_stance,cio_action,cio_confidence,cio_debate_triggered,cio_debate_conflict_level,cio_reason,cio_short_direction_match,cio_short_direction_match_reason,cio_long_direction_match,cio_long_direction_match_reason
0,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,Neutral,Watch / No Trade,0.750,True,high,"As CIO, I am ratifying the committee's unanimo...",True,matched,False,neutral missed: abnormal return moved outside ...
1,MSFT,Microsoft,2026-01-28,4.73%,-9.99%,-12.10%,-16.71%,-16.37%,-12.77%,-12.41%,...,Bullish,Accumulate on Weakness,0.850,True,high,The committee reached a unanimous post-debate ...,True,matched,False,missed
2,GOOGL,Alphabet,2026-02-03,3.59%,-1.96%,-4.96%,-9.04%,-9.36%,-5.77%,-6.11%,...,Bullish,Wait for pullback,0.750,False,low,I am adopting a final view that aligns with th...,True,matched,False,missed
3,AMZN,Amazon,2026-02-05,-8.99%,-5.55%,-7.06%,-9.67%,-7.78%,-17.79%,-16.07%,...,Bullish,Wait for pullback to initiate position,0.900,True,high,This decision overrides the deterministic guar...,True,matched,False,missed
4,META,Meta Platforms,2026-01-28,7.82%,10.40%,5.63%,-1.09%,-4.57%,6.64%,2.89%,...,Bullish,Accumulate on weakness,0.750,True,high,An override of the initial purely technical or...,False,missed,True,matched
5,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,Bearish,Short / Avoid,0.800,True,high,The decision is based on a unanimous post-deba...,True,matched,True,matched
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,Neutral,Long,0.750,True,high,This decision represents a CIO override of the...,False,missed,False,neutral missed: abnormal return moved outside ...
7,AMD,Advanced Micro Devices,2026-02-03,-6.77%,-17.31%,-13.91%,-14.94%,-17.62%,-20.69%,-23.19%,...,Bullish,Wait for Pullback,0.900,True,high,I am confirming the guardrail's stances but up...,True,matched,False,missed
8,AVGO,Broadcom,2026-03-05,2.24%,-0.69%,2.95%,-2.36%,22.42%,-0.18%,25.16%,...,Bullish,Buy,0.850,False,low,The final decision is an override of the guard...,False,missed,True,matched
9,MU,Micron Technology,2025-12-17,-8.67%,10.21%,22.65%,30.58%,94.21%,19.26%,77.38%,...,Bullish,Long,0.900,False,low,The committee reached a high-conviction bullis...,True,matched,True,matched


## Inspect Saved Outputs


In [50]:
ticker_to_inspect = "NVDA"
cached = q4.load_cached_ticker(ticker_to_inspect, CACHE_ROOT)
print("saved files:")
for key, value in q4.cached_ticker_paths(ticker_to_inspect, CACHE_ROOT).items():
    print(key, value)

packets = cached.get("packets") or []
packet_df = pd.DataFrame(packets)
display_cols = ["agent_name", "short_term_stance", "long_term_stance", "confidence", "stance_rationale"]
packet_df[[col for col in display_cols if col in packet_df.columns]]


saved files:
news_macro /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/news_macro/NVDA.json
market_technical /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/market_technical/NVDA.json
fundamental /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/fundamental/NVDA.json
packets /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/packets/NVDA.json
cio /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/NVDA.json
context /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/contexts/NVDA.json
errors /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/errors/NVDA.json


,agent_name,short_term_stance,long_term_stance,confidence,stance_rationale
0,news_macro,Bullish,Bearish,0.65,The short-term stance is Bullish due to a dire...
1,market_technical,Neutral,Bullish,0.85,The dominant technical feature is the conflict...
2,fundamental,Neutral,Bullish,0.60,NVIDIA reported an exceptionally strong quarte...


In [51]:
# Debate trace for one ticker
workflow = cio_results.get(ticker_to_inspect) or q4.load_json(CACHE_ROOT / "cio" / f"{ticker_to_inspect}.json")
pd.DataFrame(workflow["debate"].get("debate_responses", []))


,response_type,agreement,disagreement,revised_short_term_stance,revised_long_term_stance,revised_confidence,evidence_needed
0,agent_revision,{'fundamental_agent': 'I agree with the fundam...,{'market_technical_agent': 'I assign lower wei...,Bullish,Bearish,0.7,"[To invalidate the long-term bearish thesis, I..."
1,revision,{'news_macro': 'I agree with the news_macro ag...,{'news_macro': 'I assign lower weight to the n...,Bullish,Bullish,0.9,[Confirmation of a price breakout above the re...
2,agent_view_revision,[I agree with the news_macro agent's primary e...,[I disagree with the news_macro agent's long-t...,Bullish,Bullish,0.8,[Quantitative data on data center market share...


# Baseline

- always_bullish
- earnings_day_momentum：after financial report if post_1d_return > 0 then Bullish，else Bearish

In [52]:
import pandas as pd
import numpy as np

def stance_to_direction(stance):
    stance = str(stance).lower()
    if stance == "bullish":
        return "up"
    if stance == "bearish":
        return "down"
    return None

def direction_match(predicted_direction, realized_direction, abnormal_return, neutral_band=0.02):
    if predicted_direction is None or realized_direction is None or pd.isna(abnormal_return):
        return None

    abnormal_return = float(abnormal_return)

    if abs(abnormal_return) <= neutral_band:
        return predicted_direction is None

    return predicted_direction == realized_direction

def eval_baseline(
    df,
    short_stance_col,
    long_stance_col,
    prefix,
    long_post_trading_days=30,
    neutral_band=0.02,
):
    out = df.copy()

    short_pred_col = f"{prefix}_short_predicted_direction"
    long_pred_col = f"{prefix}_long_predicted_direction"
    short_match_col = f"{prefix}_short_direction_match"
    long_match_col = f"{prefix}_long_direction_match"

    long_abnormal_col = f"abnormal_{long_post_trading_days}d_vs_qqq"
    long_direction_col = f"realized_{long_post_trading_days}d_direction_vs_qqq"

    out[short_pred_col] = out[short_stance_col].apply(stance_to_direction)
    out[long_pred_col] = out[long_stance_col].apply(stance_to_direction)

    out[short_match_col] = out.apply(
        lambda row: direction_match(
            row[short_pred_col],
            row["realized_direction_vs_qqq"],
            row["abnormal_vs_qqq"],
            neutral_band=neutral_band,
        ),
        axis=1,
    )

    out[long_match_col] = out.apply(
        lambda row: direction_match(
            row[long_pred_col],
            row[long_direction_col],
            row[long_abnormal_col],
            neutral_band=neutral_band,
        ),
        axis=1,
    )

    return out

def summarize_match(df, short_col, long_col):
    short_eval = df[df[short_col].notna()]
    long_eval = df[df[long_col].notna()]

    return {
        "short_evaluable": int(len(short_eval)),
        "short_matched": int((short_eval[short_col] == True).sum()),
        "short_accuracy": float((short_eval[short_col] == True).mean()) if len(short_eval) else None,
        "long_evaluable": int(len(long_eval)),
        "long_matched": int((long_eval[long_col] == True).sum()),
        "long_accuracy": float((long_eval[long_col] == True).mean()) if len(long_eval) else None,
    }

baseline_eval = cio_eval.copy()

# Baseline 1: Always Bullish
baseline_eval["always_bullish_short_stance"] = "Bullish"
baseline_eval["always_bullish_long_stance"] = "Bullish"

baseline_eval = eval_baseline(
    baseline_eval,
    short_stance_col="always_bullish_short_stance",
    long_stance_col="always_bullish_long_stance",
    prefix="always_bullish",
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
)

# Baseline 2: Earnings-day momentum
# If the stock is up after earnings day, assume continuation; if down, assume continuation down.
baseline_eval["earnings_momentum_short_stance"] = np.where(
    baseline_eval["post_1d_return"] > 0,
    "Bullish",
    "Bearish",
)
baseline_eval["earnings_momentum_long_stance"] = baseline_eval["earnings_momentum_short_stance"]

baseline_eval = eval_baseline(
    baseline_eval,
    short_stance_col="earnings_momentum_short_stance",
    long_stance_col="earnings_momentum_long_stance",
    prefix="earnings_momentum",
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
)

baseline_summary = {
    "cio": {
        "short_accuracy": cio_summary["overall"]["cio_short_direction_match_accuracy"],
        "long_accuracy": cio_summary["overall"]["cio_long_direction_match_accuracy"],
    },
    "always_bullish": summarize_match(
        baseline_eval,
        "always_bullish_short_direction_match",
        "always_bullish_long_direction_match",
    ),
    "earnings_day_momentum": summarize_match(
        baseline_eval,
        "earnings_momentum_short_direction_match",
        "earnings_momentum_long_direction_match",
    ),
}

baseline_summary


{'cio': {'short_accuracy': 0.36363636363636365,
  'long_accuracy': 0.5454545454545454},
 'always_bullish': {'short_evaluable': 55,
  'short_matched': 28,
  'short_accuracy': 0.509090909090909,
  'long_evaluable': 55,
  'long_matched': 23,
  'long_accuracy': 0.41818181818181815},
 'earnings_day_momentum': {'short_evaluable': 55,
  'short_matched': 38,
  'short_accuracy': 0.6909090909090909,
  'long_evaluable': 55,
  'long_matched': 33,
  'long_accuracy': 0.6}}

In [53]:
bucket_rows = []

for bucket, group in baseline_eval.groupby("bucket"):
    row = {"bucket": bucket, "cases": len(group)}

    row.update({
        "cio_short_accuracy": float((group["cio_short_direction_match"] == True).mean()),
        "cio_long_accuracy": float((group["cio_long_direction_match"] == True).mean()),
    })

    row.update({
        "always_bullish_short_accuracy": float((group["always_bullish_short_direction_match"] == True).mean()),
        "always_bullish_long_accuracy": float((group["always_bullish_long_direction_match"] == True).mean()),
        "earnings_momentum_short_accuracy": float((group["earnings_momentum_short_direction_match"] == True).mean()),
        "earnings_momentum_long_accuracy": float((group["earnings_momentum_long_direction_match"] == True).mean()),
    })

    bucket_rows.append(row)

baseline_bucket_summary = pd.DataFrame(bucket_rows)
baseline_bucket_summary


,bucket,cases,cio_short_accuracy,cio_long_accuracy,always_bullish_short_accuracy,always_bullish_long_accuracy,earnings_momentum_short_accuracy,earnings_momentum_long_accuracy
0,ai_infrastructure,9,0.333333,0.555556,0.555556,0.555556,0.666667,0.666667
1,ai_semis,13,0.384615,0.615385,0.538462,0.615385,0.692308,0.692308
2,data_center_operator,2,0.000000,0.500000,1.000000,0.500000,1.000000,0.500000
3,data_center_reit,4,0.250000,0.750000,0.500000,0.500000,0.500000,0.500000
4,mega_cap_platform,7,0.714286,0.285714,0.285714,0.142857,0.857143,0.857143
5,power_infrastructure,5,0.200000,0.600000,0.800000,0.600000,0.600000,0.400000
6,software_cloud,15,0.333333,0.533333,0.400000,0.200000,0.666667,0.466667


The CIO layer adds the clearest value in AI semiconductors over the 30-trading-day horizon, where it outperforms both always-bullish and earnings-day momentum baselines. This suggests that multi-agent synthesis is most useful when post-earnings performance depends on interpreting supply-chain, capex, and second-order AI infrastructure signals. For mega-cap platform stocks, however, earnings-day momentum is a stronger baseline, implying that near-term capital flows and benchmark positioning dominate multi-agent fundamental reasoning.